In [15]:
# This notebook compares predicted masks with ground truth masks.
import cv2
import os
import numpy as np
import pandas as pd
import re

# Folders for ground-truth and predicted masks.
gt_dir = "Ground_Truth_Masks/"       
pred_dir = "detected_frames/"        

# Stop if required folders are missing.
if not os.path.exists(gt_dir) or not os.path.exists(pred_dir):
    print("❌ ERROR: Please ensure 'Ground_Truth_Masks' and 'detected_frames' exist.")
    exit()

# Store evaluation scores here.
results = []
thickness_kernel = np.ones((15, 15), np.uint8)

# Read the frame number from each filename.
def extract_frame_number(filename):
    numbers = re.findall(r'\d+', filename)
    return int(numbers[-1]) if numbers else -1

# Convert colored masks into one mask per class.
def get_color_masks(img):
    # Split the image into color channels.
    B, G, R = cv2.split(img.astype(np.int32))
    
    # Detect purple alligator damage pixels.
    alligator = ((R - G > 30) & (B - G > 30) & (np.abs(R - B) < 50)).astype(np.uint8)
    
    # Detect red pothole pixels.
    pothole = ((R - B > 30) & (R - G > 30)).astype(np.uint8)
    
    # Detect blue crack pixels.
    crack = ((B - R > 30) & (B - G > 30)).astype(np.uint8)
    
    # Avoid counting one pixel in two classes.
    pothole[alligator == 1] = 0
    crack[alligator == 1] = 0
    
    return {"Pothole": pothole, "Alligator": alligator, "Logititude Crack": crack}

# List all available mask files.
gt_files = [f for f in os.listdir(gt_dir) if f.endswith('.png')]
pred_files = [f for f in os.listdir(pred_dir) if f.endswith(('.jpg', '.png'))]

# Match prediction files by frame number.
pred_map = {extract_frame_number(f): f for f in pred_files}

print(f"Found {len(gt_files)} Ground Truth frames. Commencing Evaluation...")
frames_evaluated = 0

# Evaluate each ground-truth frame.
for fname in gt_files:
    frame_num = extract_frame_number(fname)
    
    # Skip frames that do not have predictions.
    if frame_num not in pred_map:
        print(f"⚠️ Warning: CVAT Frame {frame_num} not found in detected_frames. Skipping.")
        continue
        
    pred_path = os.path.join(pred_dir, pred_map[frame_num])
    
    # Read ground-truth and prediction images.
    gt_img = cv2.imread(os.path.join(gt_dir, fname), 1)
    pred_img = cv2.imread(pred_path, 1)
    
    if gt_img is None or pred_img is None: continue

    # Convert both images into class masks.
    gt_masks = get_color_masks(gt_img)
    pred_masks = get_color_masks(pred_img)
    
    # Compare each damage class separately.
    for class_name in ["Pothole", "Alligator", "Logititude Crack"]:
        gt_bin = gt_masks[class_name]
        pred_bin = pred_masks[class_name]
        
        # Allow small line thickness differences.
        pred_bin = cv2.dilate(pred_bin, thickness_kernel, iterations=1)

        # Count correct and wrong pixels.
        TP = np.sum((pred_bin == 1) & (gt_bin == 1))
        TN = np.sum((pred_bin == 0) & (gt_bin == 0))
        FP = np.sum((pred_bin == 1) & (gt_bin == 0))
        FN = np.sum((pred_bin == 0) & (gt_bin == 1))
        
        if TP == 0 and FP == 0 and FN == 0:
            continue
            
        # Calculate common evaluation metrics.
        acc = (TP+TN) / (TP+TN+FP+FN + 1e-6)
        iou = TP / (TP+FP+FN + 1e-6)
        prec = TP / (TP+FP + 1e-6)
        rec = TP / (TP+FN + 1e-6)
        f1 = 2 * (prec*rec) / (prec+rec + 1e-6)
        
        results.append([fname, class_name, acc, prec, rec, f1, iou])
        frames_evaluated += 1

# Print final evaluation results.
if not results:
    print("\n❌ Evaluation failed. No overlapping shapes found.")
else:
    df = pd.DataFrame(results, columns=["Frame", "Class", "Accuracy", "Precision", "Recall", "F1", "IoU"])

    print(f"\n✅ Successfully evaluated {frames_evaluated} class instances!")
    print("\n" + "="*60)
    print(" PERFORMANCE SUMMARY (AVERAGE PER CLASS)")
    print("="*60)
    # Average the scores for each class.
    summary_by_class = df.groupby("Class")[["Accuracy", "Precision", "Recall", "F1", "IoU"]].mean().round(4)
    print(summary_by_class)
    
    print("\n" + "="*60)
    print(" OVERALL SYSTEM PERFORMANCE (MACRO AVERAGE)")
    print("="*60)
    print(summary_by_class.mean().round(4))

Found 7 Ground Truth frames. Commencing Evaluation...

✅ Successfully evaluated 14 class instances!

 PERFORMANCE SUMMARY (AVERAGE PER CLASS)
                  Accuracy  Precision  Recall      F1     IoU
Class                                                        
Alligator           0.9881     0.2847  0.2938  0.2803  0.2186
Logititude Crack    0.9342     0.1199  0.5659  0.1921  0.1114
Pothole             0.9827     0.5058  0.5763  0.5285  0.3609

 OVERALL SYSTEM PERFORMANCE (MACRO AVERAGE)
Accuracy     0.9683
Precision    0.3035
Recall       0.4787
F1           0.3336
IoU          0.2303
dtype: float64
